# Transfer Learning

The question is about analyzing the effect of different **transfer learning** methods on model performance using the `CIFAR-10` dataset. We start by loading CIFAR-10 in **PyTorch** using `torchvision.datasets` and appropriate `transforms`. Two pre-trained models on ImageNet, `ResNet18` and `DenseNet121`, are selected as base models.
- In **Part A**, the models are used purely as feature extractors: all convolutional layers are **frozen**, and only a new classifier block is replaced and trained. We then evaluate performance using **Train/Validation/Test accuracy** and **loss curves**.
- In **Part B**, we move to **fine-tuning**, where some or all layers are gradually unfrozen to study how this improves performance. Finally, results are compared to understand how architecture choice, learning rate, and number of unfrozen layers affect transfer learning performance 


## Preparation

### Imports

In [ ]:
import tqdm
import torch
import random
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
from torchinfo import summary
from torch.utils.data import Subset
from torchvision import datasets, transforms, models 
from torch.utils.data import DataLoader, random_split

### GPU check

This section is mainly focused on the device we are about to use and does not interfere with other parts and can be just replaced with the following code:
```python
device = torch.device("cpu")
```

In [ ]:
!nvidia-smi

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("Using CPU")

### Data Loading

This code first defines a preprocessing pipeline (`transform`) that will be applied to every CIFAR-10 image as it’s loaded. CIFAR-10 images are originally 32×32, but here they’re resized to 96×96 (not 224×224). The key reason we *don’t* jump to 224×224 is that CIFAR-10 is very low-resolution: upscaling all the way to 224×224 would mostly add “invented” pixels (blurred/interpolated detail) rather than real information, while also making training much heavier (more memory, slower compute). Resizing to a moderate size like 96 keeps computation manageable and still makes the images a bit more compatible with common CNN backbones that expect something larger than 32×32.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(96),
    transforms.ToTensor(),
    transforms.Normalize((0.485,0.456,0.406),(0.229,0.224,0.225)),
])

Next, the datasets are loaded: `train_dataset_full` pulls the CIFAR-10 training split (50k images), and `test_dataset` pulls the test split (10k images), both with the same transform applied. The `ToTensor()` converts images into PyTorch tensors in `[0,1]`, and `Normalize((0.485,0.456,0.406),(0.229,0.224,0.225))` standardizes each RGB channel. Those specific mean/std values are the **ImageNet normalization stats**, and we use them because we’re typically doing transfer learning with models pre-trained on ImageNet (like ResNet/DenseNet). Feeding inputs normalized the same way the pre-trained model originally saw them keeps the feature distributions consistent, which usually stabilizes training and improves results compared to using arbitrary normalization.

In [ ]:
train_dataset_full = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

After loading, the code deliberately subsamples the training set: it randomly selects **30,000** examples from the full training set, then splits that subset into 80% train / 20% validation using a fixed seed for reproducibility. This is intentional because in transfer learning you often don’t need the full dataset to get strong performance—pre-trained models already provide good features—so using all 50k can just make the training process slower without a proportional gain. Finally, it builds two sets of DataLoaders with **two batch sizes**:
> a larger batch size (128) to make training faster when memory allows, and a smaller one (32) for situations where GPU/CPU memory is tighter or when you want more frequent updates. This gives flexibility to choose the batch size that best fits the hardware and training stability needs.

In [ ]:
num_samples = 30_000
indices = torch.randperm(len(train_dataset_full))[:num_samples]
small_train_dataset = Subset(train_dataset_full, indices)
train_size = int(0.8 * len(small_train_dataset))
val_size = len(small_train_dataset) - train_size
train_dataset, val_dataset = random_split(
    small_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

In [ ]:
batch_size_large = 128
train_loader_l = DataLoader(train_dataset, batch_size=batch_size_large, shuffle=True)
val_loader_l   = DataLoader(val_dataset, batch_size=batch_size_large, shuffle=False)
test_loader_l  = DataLoader(test_dataset, batch_size=batch_size_large, shuffle=False)

batch_size_small = 32
train_loader_s = DataLoader(train_dataset, batch_size=batch_size_small, shuffle=True)
val_loader_s   = DataLoader(val_dataset, batch_size=batch_size_small, shuffle=False)
test_loader_s  = DataLoader(test_dataset, batch_size=batch_size_small, shuffle=False)

### Randomization control

In [ ]:
base_seed = 42
torch.manual_seed(base_seed)
np.random.seed(base_seed)
random.seed(base_seed)
torch.cuda.manual_seed_all(base_seed)

## Model Definition

### Overview

`ImageClassifier` is a PyTorch `nn.Module` that wraps an ImageNet pre-trained backbone (either `ResNet18` or `DenseNet121`) and replaces the original classifier head with a small custom classifier for 10-class classification (CIFAR-10). The backbone’s final classification layer is swapped with `nn.Identity()` so the model outputs feature vectors (512 for ResNet18, DenseNet-dependent for DenseNet121), then a new MLP head maps features → 10 logits.

This class is designed for both feature extraction and fine-tuning. It includes a `freeze_layer()` utility that lets you control how many backbone “blocks/layers” are unfrozen. In the ResNet case, “unfreezing one block” typically means keeping most early layers frozen and allowing gradients through only the last backbone stage (the last residual block group), plus the custom classifier—so you can compare feature extraction vs gradual fine-tuning.

---

### Model Architecture

#### Backbone selection

`model_name='resnet'` → loads **ResNet18 pre-trained on ImageNet**, removes `fc` (replaced by `Identity`) and uses its feature size.

`model_name='densenet'` → loads **DenseNet121 pre-trained on ImageNet**, removes `classifier` (replaced by `Identity`) and uses its feature size.

#### Classifier head

`Linear(num_features → 64)` → `ReLU` → `Linear(64 → 10)` → `Softmax` is our selected classifier since it is complex enough to classify 10 class but not too much to cause over-fitting over the training. Also it helps us gain a probabilistic look at the classifier by using softmax at the end


#### Layer Summary 

For an instance we created a ResNet based backbone, with one block unfrozen and here is the structure and model info:

❄️ → Freezed Layer
🔥 → Trainable Layer
☢️ → None-trainable Layer

##### 🧠 Table 1 — Feature Extractor (ResNet Backbone)

| Layer                   | Output Shape      | Parameters | Freeze |
| ----------------------- | ----------------- | ---------- | ------ |
| ImageClassifier         | [1, 10]           | —          | ☢️     |
| ResNet                  | [1, 512]          | —          | ☢️     |
| Conv2d (2-1)            | [1, 64, 112, 112] | 9,408      | ❄️     |
| BatchNorm2d (2-2)       | [1, 64, 112, 112] | 128        | ❄️     |
| ReLU (2-3)              | [1, 64, 112, 112] | —          | ☢️     |
| MaxPool2d (2-4)         | [1, 64, 56, 56]   | —          | ☢️     |
| Sequential (2-5)        | [1, 64, 56, 56]   | —          | ☢️     |
| ├─ BasicBlock (3-1)     | [1, 64, 56, 56]   | 73,984     | ❄️     |
| ├─ BasicBlock (3-2)     | [1, 64, 56, 56]   | 73,984     | ❄️     |
| Sequential (2-6)        | [1, 128, 28, 28]  | —          | ☢️     |
| ├─ BasicBlock (3-3)     | [1, 128, 28, 28]  | 230,144    | ❄️     |
| ├─ BasicBlock (3-4)     | [1, 128, 28, 28]  | 295,424    | ❄️     |
| Sequential (2-7)        | [1, 256, 14, 14]  | —          | ☢️     |
| ├─ BasicBlock (3-5)     | [1, 256, 14, 14]  | 919,040    | ❄️     |
| ├─ BasicBlock (3-6)     | [1, 256, 14, 14]  | 1,180,672  | ❄️     |
| Sequential (2-8)        | [1, 512, 7, 7]    | —          | ☢️     |
| ├─ BasicBlock (3-7)     | [1, 512, 7, 7]    | 3,673,088  | 🔥     |
| ├─ BasicBlock (3-8)     | [1, 512, 7, 7]    | 4,720,640  | 🔥     |
| AdaptiveAvgPool2d (2-9) | [1, 512, 1, 1]    | —          | ☢️     |
| Identity (2-10)         | [1, 512]          | —          | ☢️     |



##### 🎯 Table 2 — Classifier Head

| Layer            | Output Shape | Parameters | Freeze |
| ---------------- | ------------ | ---------- | ------ |
| Sequential (1-2) | [1, 10]      | —          | ☢️     |
| Linear (2-11)    | [1, 64]      | 32,832     | 🔥     |
| ReLU (2-12)      | [1, 64]      | —          | ☢️     |
| Linear (2-13)    | [1, 10]      | 650        | 🔥     |
| Softmax (2-14)   | [1, 10]      | —          | ☢️     |



##### 📊 Table 3 — Model Statistics & Computational Summary

| Metric                       | Value            |
| ---------------------------- | ---------------- |
| **Total parameters**         | **11,209,994**   |
| **Trainable parameters**     | **8,427,210**    |
| **Non-trainable parameters** | **2,782,784**    |
| **Total mult-adds**          | **1.81 GFLOPs**  |



##### 💾 Table 4 — Memory & Size Breakdown

| Component                        | Size (MB)    |
| -------------------------------- | ------------ |
| **Input size**                   | 0.60 MB      |
| **Forward / Backward pass size** | 39.74 MB     |
| **Parameters size**              | 44.84 MB     |
| **Estimated total model size**   | **85.18 MB** |


---

### Functions and What They Do

`__init__(model_name='resnet', criterion=None, learning_rate=0.001, device='cpu', seed=42)`

Initializes the model:

* Sets seed for reproducibility (`torch.manual_seed(seed)`).
* Loads the chosen pretrained backbone (`resnet18` or `densenet121`).
* Removes the backbone’s original classifier layer (`fc` / `classifier`) using `nn.Identity()`.
* Builds a new classifier head for 10 classes.
* Stores the loss function (`criterion`) and creates an Adam optimizer over **all parameters**.
* Moves the whole model to the specified device.



`_get_backbone_layers()`

Returns a list of backbone layers used by `freeze_layer()` to decide what to freeze/unfreeze.

* For ResNet: returns `children()[:-2]` (everything except avgpool + fc).
* For DenseNet: returns the `features` children.
* Fallback: returns `children()`.



`freeze_layer(unfreeze_count, reset_optimizer=False)`

Controls which parts of the **backbone** train (fine-tuning) vs stay frozen (feature extraction).
Behavior:

* First sets all backbone params `requires_grad=True`.
* Then applies one of these modes:

  * `unfreeze_count == -1`: **full fine-tuning** (nothing frozen).
  * `unfreeze_count == 0`: **feature extraction** (freeze all backbone params; only classifier trains).
  * `unfreeze_count > 0`: unfreezes the **last `k` backbone layers/blocks** and freezes the earlier ones.

Optional:

* If `reset_optimizer=True`, it rebuilds Adam using only trainable parameters (useful after changing which layers are frozen).
  *(Note: your code references `self.learning_rate` but doesn’t store it in `__init__`. If you plan to use `reset_optimizer=True`, you should store `learning_rate` as `self.learning_rate = learning_rate`.)*



`forward(x)`

Defines the forward pass:

1. `x = self.base_model(x)` produces a feature vector (e.g., 512-d for ResNet18).
2. `x = self.classifier(x)` maps features to 10 output logits.



`train_one_epoch(train_loader, val_loader=None)`

Runs one training epoch:

* Sets training mode (`self.train()`).
* Iterates over the train loader:

  * moves data to device
  * computes logits → loss
  * backprop + optimizer step
* Returns average training loss.
* If `val_loader` is provided, it also computes average validation loss in `eval()` mode with `no_grad()`.

Returns:

* `(avg_train_loss, avg_val_loss)` where `avg_val_loss` can be `None`.



`predict(x, return_probs=False)`

Inference helper:

* Switches to evaluation mode.
* Accepts either a single image tensor `[C,H,W]` or batch `[N,C,H,W]`.
* Returns predicted class indices (`argmax`).
* If `return_probs=True`, returns `(preds, probs)` where `probs` is the raw output tensor from the model (logits in this code).



`set_learning_rate(new_lr)`

Updates learning rate inside the optimizer parameter groups.



`get_learning_rate()`

Returns current learning rate from the optimizer.



`set_device(device)`

Moves the model to a new device (string like `"cuda"` or a `torch.device`).



`get_device()`

Returns the current device used by the model.



`summarize(input_size=(3, 224, 224))`

Calls `torchinfo.summary()` for a clean model summary given an input size (default assumes ImageNet-like 224×224).
If you trained on smaller resized inputs (e.g., 96×96), you can pass `input_size=(3, 96, 96)` to match your pipeline.


In [ ]:
class ImageClassifier(nn.Module):  
    def __init__(self, model_name='resnet', criterion=None, learning_rate=0.001, device='cpu', seed=42):
        torch.manual_seed(seed)
        super(ImageClassifier, self).__init__()  

        if isinstance(device, str):  
            device = torch.device(device)  
        self.device = device 
        self.base_model_name = model_name.lower()
        if self.base_model_name == 'resnet':  
            base_model = models.resnet18(pretrained=True)
            num_features = base_model.fc.in_features  
            base_model.fc = nn.Identity()
        elif self.base_model_name == 'densenet':
            base_model = models.densenet121(pretrained=True)  
            num_features = base_model.classifier.in_features  
            base_model.classifier = nn.Identity()
        else:  
            raise ValueError(f"Unsupported model_name: {model_name}")  
        self.base_model = base_model

        self.classifier = nn.Sequential(
            nn.Linear(num_features, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 10),
        )

        self.criterion = criterion  
        self.optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate)  
        self.to(self.device)  

    
    def _get_backbone_layers(self):
        if self.base_model_name.startswith('resnet'):
            return list(self.base_model.children())[:-2]

        if self.base_model_name.startswith('densenet'):
            return list(self.base_model.features.children())

        return list(self.base_model.children())

    def freeze_layer(self, unfreeze_count, reset_optimizer=False):
        backbone_layers = self._get_backbone_layers()
        for param in self.base_model.parameters():
            param.requires_grad = True
        if unfreeze_count == -1:
            pass

        elif unfreeze_count == 0:
            for param in self.base_model.parameters():
                param.requires_grad = False
        else:
            num_backbone_layers = len(backbone_layers)
            k = int(unfreeze_count)
            if k < 0:
                k = 0
            if k > num_backbone_layers:
                k = num_backbone_layers
            freeze_upto = num_backbone_layers - k
            for layer in backbone_layers[:freeze_upto]:
                for param in layer.parameters():
                    param.requires_grad = False
        if reset_optimizer:
            self.optimizer = torch.optim.Adam(
                (p for p in self.parameters() if p.requires_grad),
                lr=self.learning_rate
            )

    def forward(self, x):
        x = self.base_model(x) 
        x = self.classifier(x)
        return x  

    def train_one_epoch(self, train_loader, val_loader=None):
        self.train()  
        total_train_loss = 0.0  
        num_batches = 0  
        for inputs, labels in train_loader:  
            inputs = inputs.to(self.device)
            labels = labels.to(self.device)  

            outputs = self(inputs)
            loss = self.criterion(outputs, labels)
            self.optimizer.zero_grad()  
            loss.backward()
            self.optimizer.step()
            total_train_loss += loss.item()  
            num_batches += 1  
        avg_train_loss = total_train_loss / num_batches if num_batches > 0 else 0.0  

        avg_val_loss = None  
        if val_loader is not None:  
            self.eval()  
            total_val_loss = 0.0  
            val_batches = 0  
            with torch.no_grad():
                for inputs, labels in val_loader:  
                    inputs = inputs.to(self.device)  
                    labels = labels.to(self.device)  
                    outputs = self(inputs)
                    loss = self.criterion(outputs, labels)  
                    total_val_loss += loss.item()  
                    val_batches += 1  
            avg_val_loss = total_val_loss / val_batches if val_batches > 0 else 0.0  

        return avg_train_loss, avg_val_loss  

    def predict(self, x, return_probs=False):
        self.eval()
        with torch.no_grad():
            x = x.to(self.device)
            if x.dim() == 3:
                x = x.unsqueeze(0)
            outputs = self(x)
            probs = outputs
            preds = torch.argmax(probs, dim=1)
        if return_probs:
            return preds, probs
        return preds


    def set_learning_rate(self, new_lr):
        for param_group in self.optimizer.param_groups:  
            param_group['lr'] = new_lr  

    def get_learning_rate(self):
        return self.optimizer.param_groups[0]['lr']

    def set_device(self, device):
        if isinstance(device, str):  
            device = torch.device(device)  
        self.device = device  
        self.to(self.device) 

    def get_device(self):
        return self.device  

    def summarize(self, input_size=(3, 224, 224)):
        return summary(self, input_size=(1, *input_size), device=self.device)

In [ ]:
model = ImageClassifier(model_name='resnet', criterion=nn.CrossEntropyLoss(), device=device)

In [ ]:
model.freeze_layer(0)
print(model.summarize())
model.freeze_layer(1)
print(model.summarize())
model.freeze_layer(-1)
print(model.summarize())

## Evaluation

### Metrics

The code defines `compute_accuracy`, which evaluates a classification model’s performance by computing **accuracy** over an entire dataloader. It switches the model to evaluation mode (`model.eval()`) so layers like dropout and batch normalization behave deterministically, and it disables gradient tracking (`torch.no_grad()`) to make evaluation faster and more memory-efficient. For each batch, it gets the true labels, runs the model’s `predict` method to obtain predicted class labels, compares predictions to the ground truth, counts how many are correct, and divides by the total number of samples to return the final accuracy (returning `0.0` safely if the dataloader is empty).

In [ ]:
def compute_accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            labels = labels.to(model.device)
            preds = model.predict(inputs)
            preds = preds.to(model.device)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    return correct / total if total > 0 else 0.0



### Plots

This code defines `plot_train_val`, which produces a line plot showing how a metric (such as loss or accuracy) changes across epochs for both the **training** and **validation** sets. It creates an epoch index from 1 to the number of recorded points, plots the two curves with labels, adds axis labels and a title, displays a legend, and enables a grid for readability. This makes it easy to visually diagnose training behavior—like steady improvement, overfitting (train improves while validation worsens), or underfitting (both curves stay poor).

In [ ]:
def plot_train_val(train_values, val_values, title):
    epochs = range(1, len(train_values) + 1)
    plt.figure()
    plt.plot(epochs, train_values, label="Train")
    plt.plot(epochs, val_values, label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Value")
    plt.title(title)
    plt.legend()
    plt.grid(True)
    plt.show()


## Answers to Homework

### Part A

#### Experimental setup

A transfer-learning experiment was conducted using two pre-trained backbones—**DenseNet** and **ResNet**—as fixed (frozen) feature extractors, with a custom classification head trained on top. The backbone parameters were kept non-trainable (`unfreeze_count=0`), so optimization updated only the classifier head. Training ran for **15 epochs** with **CrossEntropyLoss**. The learning rate started at **0.005**, was reduced to **0.002 at epoch 6**, and reduced again to **0.001 at epoch 11**.

During training, the same underlying datasets were used throughout; however, at epoch 11 the **batch size was reduced** (switching from `*_l` loaders to `*_s` loaders) to obtain **more stable/accurate mini-batch estimates** of loss/accuracy (i.e., lower variance in the batch statistic itself), at the cost of noisier gradients and longer epoch time.

At each epoch, the script logged **training loss**, **validation loss**, and **validation accuracy**. After training, it reported accuracy on the full **Train**, **Validation**, and **Test** splits and plotted **training vs. validation loss** over epochs.


#### Reported results
##### DenseNet (frozen backbone)

- **Final accuracies**
  - Train: **0.8955**
  - Validation: **0.7880**
  - Test: **0.7838**

- **Best observed validation accuracy (during training)**
  - Peak validation accuracy ≈ **0.7965** (epoch 9)

- **Loss trend**
  - Training loss decreased from **0.7970 → 0.3521** by epoch 10, then rose to **0.5092** at epoch 11 and gradually decreased again to **0.4417** by epoch 15.
  - Validation loss remained relatively flat and noisy, roughly **0.62–0.67** across epochs.

![Loss](.\Output\Transfer\PartA\densenet.png)

##### ResNet (frozen backbone)

- **Final accuracies**
  - Train: **0.8528**
  - Validation: **0.7570**
  - Test: **0.7556**

- **Best observed validation accuracy (during training)**
  - Peak validation accuracy ≈ **0.7660** (epoch 6)

- **Loss trend**
  - Training loss decreased from **0.9294 → 0.5029** by epoch 10, then rose to **0.6304** at epoch 11 and ended at **0.5803** by epoch 15.
  - Validation loss remained high and fairly flat, around **0.71–0.79**, with mild fluctuations.

![Loss](.\Output\Transfer\PartA\resnet.png)

#### What we have learned

> Why training loss jumps around epoch 11

At epoch 11 the learning rate is reduced **and** the batch size is reduced. Even when the underlying dataset is unchanged, **changing batch size changes the optimization dynamics**:

- **Smaller batches produce noisier gradients**, which can cause temporary increases or oscillations in training loss, especially immediately after the change.
- If the reported “train loss” is computed as an **average of batch losses** (common in training loops), then changing batch size can also change how representative each batch is of the full data distribution, leading to a visible shift in the per-epoch average.
- The loss spike at epoch 11 is therefore consistent with a **batch-size transition effect**, not necessarily model collapse.

> Is it Always wrong to change batch sizes?

Reducing the batch size later in training is **not necessarily a bad design choice**, especially in a practical training setup. Using a larger batch size (128) in the early epochs allows the model to **train faster and more efficiently**, since each epoch requires fewer parameter updates and better utilizes hardware resources. After the model has largely converged, switching to a smaller batch size (32) can provide **slightly noisier but more detailed gradient updates**, which may help refine the solution and improve generalization. While this approach is not guaranteed to improve performance in all cases, it represents a **reasonable trade-off between training speed and optimization precision** rather than an arbitrary or flawed decision.


> Evidence of overfitting and why it happens here

Both backbones show signs of **overfitting (or limited generalization)**:

- In DenseNet, the final **Train accuracy (0.8955)** is much higher than **Validation/Test (~0.788/~0.784)**.
- In ResNet, the final **Train accuracy (0.8528)** is also higher than **Validation/Test (~0.757/~0.756)**.
- Training loss decreases substantially, but **validation loss does not decrease in the same way** and remains relatively flat.

This behavior is expected in a **frozen-feature-extractor** setup for two main reasons:

1. **The classifier head can over-specialize to the training set.**  
   With the backbone frozen, the model cannot adapt low- and mid-level visual features to match the target dataset. The head compensates by fitting decision boundaries that work well on the training distribution, which can reduce training loss without improving validation loss.

2. **Representation mismatch (ImageNet → CIFAR-10).**  
   Pre-trained backbones are typically trained on ImageNet-sized natural images. When applied to smaller, lower-resolution datasets (like CIFAR-10), the frozen features may not be perfectly aligned with the target task. Without fine-tuning, the model may hit a generalization ceiling.

The relatively stable validation accuracy (DenseNet mostly ~0.77–0.80; ResNet mostly ~0.74–0.77) suggests the models reach a plateau where the head has extracted most of what it can from fixed features.


> how suitable are ImageNet features for CIFAR-10?

Based on the obtained results, ImageNet's pre-trained features are **reasonably suitable but not perfectly aligned** with CIFAR-10. Both DenseNet and ResNet achieve **moderate to good validation and test accuracy (≈75–79%)** using frozen backbones, which indicates that ImageNet features capture **general visual patterns** such as edges, textures, and simple object parts that transfer well to CIFAR-10. However, the clear gap between training accuracy and validation/test accuracy, along with the early plateau in validation performance, suggests that these features are **not fully optimal for the CIFAR-10 domain** when kept frozen. CIFAR-10 images are much smaller and less diverse than ImageNet images, so without fine-tuning, the pretrained representations impose a performance ceiling. Overall, the results show that ImageNet features provide a **strong and effective starting point**, but **fine-tuning is necessary** to fully adapt them to CIFAR-10 and achieve better generalization.


#### Comparative conclusion

- **DenseNet outperforms ResNet** in this configuration:
  - Validation: **0.788 vs 0.757**
  - Test: **0.7838 vs 0.7556**
- DenseNet also reaches a **higher peak validation accuracy** during training (≈0.7965 vs ≈0.7660).
- Both models exhibit a generalization gap, indicating that **fine-tuning** (unfreezing later blocks) is likely needed to improve validation/test performance further.
- What we recommend here is to **Add regularization to the head**: dropout, weight decay, or label smoothing to reduce overfitting. 

In [ ]:
num_epochs = 15
for base_model in ['densenet', 'resnet']:
    print(f'Training with {base_model} as the base feature extraction model')
    model = ImageClassifier(
        model_name=base_model,
        criterion=nn.CrossEntropyLoss(),
        learning_rate=0.005,
        device=device,
        seed=base_seed
    )
    model.freeze_layer(unfreeze_count=0)
    train_loader = train_loader_l
    val_loader = val_loader_l
    train_losses, val_losses = [], []
    for epoch in range(num_epochs):
        if epoch == 5:
            model.set_learning_rate(0.002)
        elif epoch == 10:
            model.set_learning_rate(0.001)
            train_loader = train_loader_s
            val_loader = val_loader_s
        t_loss, v_loss = model.train_one_epoch(train_loader, val_loader)
        train_losses.append(t_loss)
        val_losses.append(v_loss)
        print(f'epoch:{epoch+1} ====== (train loss: {t_loss:.4f}, validation loss: {v_loss:.4f}) ====== Validation accuracy: {compute_accuracy(model, val_loader)}')
    print('Model accuracy for:') 
    print(f'Train = {compute_accuracy(model, train_loader_l)}')
    print(f'Test = {compute_accuracy(model, test_loader_l)}')
    print(f'Validation = {compute_accuracy(model, val_loader_l)}')
    plot_train_val(train_losses, val_losses, title="Training vs Validation Loss")

### Part B

#### Experimental Overview

This experiment investigates the effect of **progressively unfreezing pretrained backbone layers** on model performance for an image classification task. Two pre-trained architectures, **DenseNet121** and **ResNet18**, were used as base feature extractors initialized with ImageNet weights. Unlike earlier experiments where the backbone was kept fully frozen, this study focuses exclusively on **fine-tuning scenarios**, since the fully frozen case had already been evaluated previously.

For each backbone, three configurations were examined:
- **Unfreeze 1 block** (only the last block trainable)
- **Unfreeze 3 blocks**
- **Unfreeze all layers** (full fine-tuning)

All models were trained for **15 epochs** using cross-entropy loss. The initial learning rate was set to **0.001**, and reduced to **0.0002 at epoch 11** to stabilize training during later stages. The same training, validation, and test datasets were used throughout. Also, The full-freezed feature extractors have been tested before, therefore we prefer not to waste time on it

---

#### Results Summary

##### DenseNet121

| Trainable Blocks | Train Acc | Val Acc | Test Acc |
|------------------|----------|---------|----------|
| 1 block | 0.907 | 0.821 | 0.8256 |
| 3 blocks | 1.000 | 0.9135 | 0.9103 |
| All layers | 1.000 | **0.9315** | **0.9286** |

DenseNet shows a **clear and consistent improvement** as more layers are unfrozen. Allowing only one block to adapt already improves performance compared to a frozen backbone, but unfreezing three blocks leads to a substantial gain in validation and test accuracy. Fully fine-tuning the network achieves the **best overall performance**, indicating that DenseNet benefits strongly from adapting its higher- and mid-level features to the target dataset.

##### ResNet18

| Trainable Blocks | Train Acc | Val Acc | Test Acc |
|------------------|----------|---------|----------|
| 1 block | 1.000 | 0.895 | 0.8897 |
| 3 blocks | 1.000 | **0.9025** | **0.9001** |
| All layers | 1.000 | 0.8952 | 0.8945 |

For ResNet, unfreezing layers also improves performance compared to limited fine-tuning, but the gains are **more modest** than those observed for DenseNet. The best validation and test performance occurs when **three blocks are unfrozen**, while fully unfreezing the network does not yield additional improvements and slightly degrades generalization.

---


#### Loss Curve Visualizations

The following figures illustrate the relationship between training and validation loss for each backbone and fine-tuning configuration:

##### DenseNet121
**Figure 1:** DenseNet with 1 unfrozen block<br>
![Loss](.\Output\Transfer\PartB\1densenet.png)<br>
**Figure 2:** DenseNet with 3 unfrozen blocks<br>
![Loss](.\Output\Transfer\PartB\3densenet.png)<br>
**Figure 3:** DenseNet with all layers trainable<br>
![Loss](.\Output\Transfer\PartB\fulldensenet.png) 

##### ResNet18
**Figure 4:** ResNet with 1 unfrozen block<br> 
![Loss](.\Output\Transfer\PartB\1resnet.png)<br>
**Figure 5:** ResNet with 3 unfrozen blocks<br>
![Loss](.\Output\Transfer\PartB\3resnet.png)<br>
**Figure 6:** ResNet with all layers trainable<br>
![Loss](.\Output\Transfer\PartB\fullresnet.png) 

These plots visually support the numerical results by highlighting how increased fine-tuning depth accelerates training loss reduction while making validation behavior more variable, reinforcing the trade-off between adaptability and generalization.


1. the validation loss behaves differently depending on the fine-tuning depth. When only **one block is unfrozen**, both DenseNet and ResNet show relatively smooth and stable validation loss curves that decrease slowly and then plateau. This suggests controlled learning, where only high-level features are adapted, limiting overfitting while still improving performance compared to a fully frozen backbone.

2. When **three blocks are unfrozen**, training loss drops much faster and validation loss becomes more oscillatory. These fluctuations indicate increased sensitivity to the training data, as deeper feature representations are now updated. Despite this instability, validation accuracy improves substantially, showing that the additional flexibility allows the model to better align ImageNet features with the target dataset.

3. In the **fully fine-tuned** setting, training loss rapidly approaches zero, often within the first half of training. While this demonstrates very strong fitting ability, validation loss shows noticeable oscillations and occasional increases. This behavior reflects a higher risk of overfitting, as the model begins to memorize training patterns rather than learning features that generalize consistently. Nevertheless, DenseNet benefits more from full fine-tuning than ResNet, achieving the lowest validation loss and highest validation accuracy overall.

Overall, the loss curves confirm that progressively unfreezing layers increases model capacity and adaptation ability, but also introduces instability and overfitting risk. Partial fine-tuning offers a balance between stability and performance, while full fine-tuning must be applied carefully and validated empirically.


#### Effect of Progressive Unfreezing

The results demonstrate that **progressive unfreezing has a strong impact on model performance**, but the optimal depth of fine-tuning depends on the architecture:

- **Unfreezing a small number of blocks** allows the model to adapt high-level semantic features to the new dataset, leading to noticeable improvements over frozen backbones.
- **Unfreezing more blocks** generally improves performance further by allowing deeper feature adaptation.
- **Full fine-tuning** can yield the best results (as seen with DenseNet), but it also increases the risk of overfitting and optimization instability (as observed with ResNet).

In both architectures, training accuracy reaches 100% when multiple blocks are unfrozen, indicating high model capacity. However, validation and test accuracy reveal that **more trainable layers do not always guarantee better generalization**, emphasizing the importance of controlled fine-tuning.

---

#### Conclusion

This study shows that ImageNet's pre-trained models benefit significantly from **partial and progressive fine-tuning** when transferred to a new dataset. DenseNet exhibits strong gains from full fine-tuning, achieving the highest validation and test accuracy overall. ResNet, on the other hand, performs best with **intermediate fine-tuning**, where only the last few blocks are unfrozen.

Overall, the results confirm that:
- Fully frozen backbones limit performance.
- Gradual unfreezing improves adaptability and accuracy.
- The optimal fine-tuning depth is **architecture-dependent** and must be validated experimentally.

These findings support the use of **progressive unfreezing as an effective and principled transfer learning strategy**, rather than relying solely on either frozen features or full fine-tuning.


In [ ]:
num_epochs = 15
for base_model in ['densenet', 'resnet']:
    for unfreeze in [1, 3, -1]:
        print(f'Training with {base_model} as the base feature extraction model.')
        print('All layers are trainable' if unfreeze == -1 else f'{unfreeze} block of the base model is/are trainable')
        model = ImageClassifier(
            model_name=base_model,
            criterion=nn.CrossEntropyLoss(),
            learning_rate=0.001,
            device=device,
            seed=base_seed
        )
        model.freeze_layer(unfreeze_count=unfreeze)
        train_loader = train_loader_l
        val_loader = val_loader_l
        train_losses, val_losses = [], []
        for epoch in range(num_epochs):
            if epoch == 10:
                model.set_learning_rate(0.0002)
            t_loss, v_loss = model.train_one_epoch(train_loader, val_loader)
            train_losses.append(t_loss)
            val_losses.append(v_loss)
            print(f'epoch:{epoch+1} ====== (train loss: {t_loss:.4f}, validation loss: {v_loss:.4f}) ====== Validation accuracy: {compute_accuracy(model, val_loader)}')
        print('Model accuracy for:') 
        print(f'Train = {compute_accuracy(model, train_loader_l)}')
        print(f'Test = {compute_accuracy(model, test_loader_l)}')
        print(f'Validation = {compute_accuracy(model, val_loader_l)}')
        plot_train_val(train_losses, val_losses, title="Training vs Validation Loss")

### Questions

#### 1. Effect of Learning Rate on Transfer Learning Performance

The learning rate plays a critical role in transfer learning by controlling how strongly pre-trained features are adapted to the target dataset. In the conducted experiments, higher learning rates enabled faster initial convergence but risked destabilizing training when many pre-trained layers were unfrozen. Conversely, lower learning rates provided more stable updates, especially during fine-tuning.

The results show that **learning rate reduction during training was essential for achieving optimal performance**. In both frozen and fine-tuned settings, decreasing the learning rate in later epochs led to smoother convergence and improved validation accuracy. This effect was particularly pronounced when multiple backbone blocks or the full network were unfrozen, as smaller learning rates prevented large, destructive updates to pre-trained weights.

The optimal learning rate was achieved empirically by:
- Starting with a moderate value to allow effective adaptation,
- Monitoring validation loss and accuracy,
- Gradually reducing the learning rate once performance plateaued.

This strategy allowed the models to balance adaptation and stability, resulting in improved generalization.

---

#### 2. Effect of the Number of Frozen Layers on Model Performance

The number of frozen layers had a **direct and significant impact** on model performance.

- With a **fully frozen backbone** (evaluated previously), the models achieved moderate accuracy, indicating that ImageNet features provide a strong baseline but impose a performance ceiling.
- **Unfreezing one block** consistently improved validation and test accuracy by allowing high-level semantic features to adapt to the new dataset while maintaining training stability.
- **Unfreezing three blocks** produced a substantial performance gain for both architectures, showing that deeper feature adaptation is highly beneficial.
- **Fully unfreezing the backbone** yielded the best results for DenseNet but showed diminishing returns for ResNet, with signs of increased overfitting.

These findings demonstrate that progressive unfreezing increases representational flexibility and accuracy, but excessive unfreezing can introduce instability and overfitting. Therefore, the optimal number of trainable layers depends on both the dataset and the architecture.

---

#### 3. Effect of Model Architecture on Transfer Learning

Model architecture strongly influenced transfer learning effectiveness.

Across all configurations, **DenseNet consistently outperformed ResNet** in validation and test accuracy. DenseNet benefited more from progressive unfreezing and achieved its highest performance when fully fine-tuned. This suggests that DenseNet’s dense connectivity pattern enables more effective feature reuse and smoother gradient flow during fine-tuning.

ResNet also benefited from unfreezing layers, but its optimal performance occurred when **only part of the network was fine-tuned**. Fully unfreezing ResNet did not yield additional improvements and slightly reduced generalization, indicating that its residual structure may be more sensitive to over-updating pre-trained weights.

Overall, the results indicate that:
- DenseNet is more adaptable to full fine-tuning,
- ResNet performs best with controlled, partial fine-tuning,
- Architecture choice directly affects how aggressively a model should be fine-tuned.

---

#### 4. Effectiveness of Transfer Learning in Increasing Accuracy

Transfer learning proved to be **highly effective** in improving model performance.

Compared to training only a classifier on frozen features, progressive fine-tuning led to **large accuracy gains**, increasing validation accuracy from the high 70% range to over **93%** in the best configuration. These improvements were achieved without training models from scratch, significantly reducing computational cost and training time.

The results confirm that:
- ImageNet-pre-trained features provide a strong starting point,
- Fine-tuning allows these features to adapt meaningfully to the target dataset,
- Transfer learning enables high accuracy even with limited data and training time.

In conclusion, transfer learning substantially enhanced performance, with progressive unfreezing and careful learning rate scheduling emerging as key factors in achieving optimal results.

---

#### Final Conclusion

This study demonstrates that transfer learning performance is governed by the interaction between learning rate, layer freezing strategy, and model architecture. Optimal results were achieved by combining moderate initial learning rates, scheduled reductions, and controlled fine-tuning depth tailored to the backbone architecture. DenseNet proved more amenable to full fine-tuning, while ResNet required a more conservative approach. Overall, transfer learning was shown to be an effective and efficient strategy for significantly improving classification accuracy.
